<!-- paired-role-banner -->
> 🟩 **정답 코드 · 오른쪽 창** — 왼쪽과 같은 셀 위치의 참고 구현입니다. 필요한 한 줄만 확인하고 다시 실습 창으로 돌아가세요.

# 14. RNN/GRU/LSTM 시퀀스 모델링

센서 시간 창을 이용해 **이상 징후 분류**와 **다음 온도 예측**을 수행합니다.
vanilla RNN, GRU, LSTM의 인터페이스를 같은 클래스로 비교하고, 가변 길이 padding,
`pack_padded_sequence`, hidden state shape, gradient clipping, checkpoint까지 연습합니다.

In [ ]:
# 이 컴퓨터를 감지해 안전한 시작값을 확인합니다. 각 실습은 더 작은 값으로도 동작합니다.
from llm_engineering_lab.hardware import HardwareProfile, recommend_training_profile

LAB_HARDWARE = HardwareProfile.detect()
LAB_PROFILE = recommend_training_profile(LAB_HARDWARE)
print("hardware:", LAB_HARDWARE)
print("recommended profile:", LAB_PROFILE)

## 실행 규모

기본 설정은 CPU에서도 수 분 이내, RTX 3080에서는 더 빠르게 끝나도록 작은 모델과
적은 epoch를 사용합니다. GPU 메모리보다 데이터 전송 시간이 더 큰 규모이므로
`device` 코드는 연습하되, CPU가 반드시 열등한 선택은 아닙니다.

<!-- api-explanations:start -->
<!-- api-explanations:keys=random.seed,np.random.seed,torch.manual_seed,torch.cuda.manual_seed_all -->
<details>
<summary><strong>이 셀에서 처음 만나는 함수·API·수식 해설 (API 4개 · 수식 0개)</strong></summary>

> 아래 내용은 라이브러리 API의 의미를 설명합니다. 실습 정답 로직은 포함하지 않습니다.
> 수식은 별도 표기가 없으면 대표적인 기본 설정을 나타냅니다. `dim`, `reduction`, 가중치 같은 실제 인자에 따라 세부 형태가 달라질 수 있습니다.

<details>
<summary><code>random.seed(a=None)</code></summary>

#### `random.seed(a=None)`

- **역할:** Python 표준 난수 생성기의 상태를 초기화합니다.
- **입력·반환:** 정수 등의 시드를 받고 반환값은 없습니다.
- **이 셀에서 쓰는 이유:** 같은 Python 난수 수열을 다시 만들어 실험을 재현합니다.
- **주의점:** NumPy와 PyTorch의 난수 상태는 별도로 고정해야 합니다.
- **공식 문서:** [random.seed](https://docs.python.org/3/library/random.html#random.seed)

</details>

<details>
<summary><code>np.random.seed(seed=None)</code></summary>

#### `np.random.seed(seed=None)`

- **역할:** NumPy의 기존 전역 난수 생성기 상태를 초기화합니다.
- **입력·반환:** 시드를 받고 반환값은 없습니다.
- **이 셀에서 쓰는 이유:** 레거시 `np.random.*` 호출의 수열을 재현합니다.
- **주의점:** 새 코드에서는 독립 상태를 가진 `default_rng`가 권장됩니다.
- **공식 문서:** [np.random.seed](https://numpy.org/doc/stable/reference/random/generated/numpy.random.seed.html)

</details>

<details>
<summary><code>torch.manual_seed(seed)</code></summary>

#### `torch.manual_seed(seed)`

- **역할:** PyTorch 난수 생성기의 시드를 설정합니다.
- **입력·반환:** 정수 시드를 받고 `Generator`를 반환합니다.
- **이 셀에서 쓰는 이유:** 가중치 초기화와 텐서 샘플링을 반복 가능하게 만듭니다.
- **주의점:** 완전한 결정성에는 알고리즘·장치 설정도 추가로 필요할 수 있습니다.
- **공식 문서:** [torch.manual_seed](https://docs.pytorch.org/docs/stable/generated/torch.manual_seed.html)

</details>

<details>
<summary><code>torch.cuda.manual_seed_all(seed)</code></summary>

#### `torch.cuda.manual_seed_all(seed)`

- **역할:** 사용 가능한 모든 CUDA GPU의 난수 시드를 설정합니다.
- **입력·반환:** 정수 시드를 받고 반환값은 없습니다.
- **이 셀에서 쓰는 이유:** 여러 GPU에서도 초기화와 샘플링을 최대한 반복 가능하게 합니다.
- **주의점:** CPU 시드와 결정론 설정을 대신하지 않습니다.
- **공식 문서:** [torch.cuda.manual_seed_all](https://docs.pytorch.org/docs/stable/generated/torch.cuda.manual_seed_all.html)

</details>

</details>
<!-- api-explanations:end -->

In [ ]:
import copy
import random
from dataclasses import dataclass

import numpy as np
import pandas as pd
import torch
from sklearn.metrics import f1_score, mean_absolute_error
from torch import nn
from torch.nn.utils import clip_grad_norm_
from torch.nn.utils.rnn import pack_padded_sequence, pad_sequence
from torch.utils.data import DataLoader, Dataset
from llm_engineering_lab.acceleration import get_accelerator

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
ACCELERATOR = get_accelerator()
device = ACCELERATOR.device
print(ACCELERATOR.summary(), "torch:", torch.__version__)

In [ ]:
from pathlib import Path


def find_project_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for candidate in (start, *start.parents):
        if (candidate / "data" / "practice").exists():
            return candidate
    raise FileNotFoundError(
        "프로젝트 루트를 찾지 못했습니다. Start_JupyterLab.cmd로 실행해 주세요."
    )


ROOT = find_project_root()
DATA = ROOT / "data" / "practice"
ARTIFACTS = ROOT / "artifacts" / "practice"
ARTIFACTS.mkdir(parents=True, exist_ok=True)
print("project root:", ROOT)

## 1) 시간 순서 보존과 train 통계

장비별 앞 75%를 train, 뒤 25%를 test로 두고, 표준화 통계는 train 구간만으로
계산합니다. target window 안에 anomaly가 하나라도 있으면 양성으로 정의합니다.

<!-- api-explanations:start -->
<!-- api-explanations:keys=pd.read_csv,Tensor.mean -->
<details>
<summary><strong>이 셀에서 처음 만나는 함수·API·수식 해설 (API 2개 · 수식 1개)</strong></summary>

> 아래 내용은 라이브러리 API의 의미를 설명합니다. 실습 정답 로직은 포함하지 않습니다.
> 수식은 별도 표기가 없으면 대표적인 기본 설정을 나타냅니다. `dim`, `reduction`, 가중치 같은 실제 인자에 따라 세부 형태가 달라질 수 있습니다.

<details>
<summary><code>pd.read_csv(filepath_or_buffer, ...)</code></summary>

#### `pd.read_csv(filepath_or_buffer, ...)`

- **역할:** CSV 표 데이터를 DataFrame으로 읽습니다.
- **입력·반환:** 표 데이터와 열·그룹 설정을 받고 DataFrame, Series 또는 집계 객체를 반환합니다.
- **이 셀에서 쓰는 이유:** 학습 전 데이터 분포·결측·중복과 분할 기준을 점검합니다.
- **주의점:** dtype 추론, 결측값, index 보존과 grouping 후 shape 변화를 확인해야 합니다.
- **공식 문서:** [pd.read_csv](https://pandas.pydata.org/docs/reference/api/pandas.read_csv.html)

</details>

<details>
<summary><code>tensor.mean(dim=None, keepdim=False, dtype=None)</code></summary>

#### `tensor.mean(dim=None, keepdim=False, dtype=None)`

- **역할:** 지정 축의 산술평균을 계산합니다.
- **입력·반환:** 텐서와 축 설정을 받고 축약된 텐서를 반환합니다.
- **이 셀에서 쓰는 이유:** batch loss, feature 통계, metric을 하나의 대표값으로 축약합니다.
- **주의점:** 어느 축을 없애는지와 빈 텐서에서 NaN이 나올 수 있음을 확인합니다.

##### 관련 수식과 코드 연결

$$
\bar x=\frac{1}{N}\sum_{i=1}^{N}x_i
$$

- **기호:** $N$은 선택한 축의 원소 수입니다.
- **수식 → 코드:** `dim` 또는 `axis`가 평균에 포함할 원소와 결과에서 사라질 축을 결정합니다.
- **직관:** 여러 관측값을 중심 위치 하나로 요약합니다.
- **shape 확인:** `keepdim=False`이면 축약한 축이 제거되고 참이면 크기 1로 남습니다.
- **공식 문서:** [Tensor.mean](https://docs.pytorch.org/docs/stable/generated/torch.mean.html)

</details>

</details>
<!-- api-explanations:end -->

In [ ]:
frame = pd.read_csv(DATA / "sensor_timeseries.csv", parse_dates=["timestamp"])
FEATURES = ["load", "temperature", "vibration", "pressure"]

train_parts, test_parts = [], []
for _, group in frame.sort_values("timestamp").groupby("device_id", sort=False):
    cut = int(len(group) * 0.75)
    train_parts.append(group.iloc[:cut])
    test_parts.append(group.iloc[cut:])
train_frame = pd.concat(train_parts, ignore_index=True)
test_frame = pd.concat(test_parts, ignore_index=True)
means = train_frame[FEATURES].mean()
stds = train_frame[FEATURES].std().replace(0, 1.0)
print(train_frame.shape, test_frame.shape, "raw anomaly rate=", frame.anomaly.mean())

<!-- api-explanations:start -->
<!-- api-explanations:keys=torch.tensor,torch.float32,Tensor.sum -->
<details>
<summary><strong>이 셀에서 처음 만나는 함수·API·수식 해설 (API 3개 · 수식 1개)</strong></summary>

> 아래 내용은 라이브러리 API의 의미를 설명합니다. 실습 정답 로직은 포함하지 않습니다.
> 수식은 별도 표기가 없으면 대표적인 기본 설정을 나타냅니다. `dim`, `reduction`, 가중치 같은 실제 인자에 따라 세부 형태가 달라질 수 있습니다.

<details>
<summary><code>torch.tensor(data, dtype=None, device=None)</code></summary>

#### `torch.tensor(data, dtype=None, device=None)`

- **역할:** 데이터를 복사해 새 PyTorch 텐서를 만듭니다.
- **입력·반환:** 배열형 데이터와 선택적 dtype·device를 받고 텐서를 반환합니다.
- **이 셀에서 쓰는 이유:** 상수·label·축소 실험 입력을 명시적으로 구성합니다.
- **주의점:** 기존 텐서에서 호출하면 복사가 생기므로 경우에 따라 `as_tensor`가 낫습니다.
- **공식 문서:** [torch.tensor](https://docs.pytorch.org/docs/stable/generated/torch.tensor.html)

</details>

<details>
<summary><code>torch.float32</code></summary>

#### `torch.float32`

- **역할:** 32비트 부동소수점 dtype
- **입력·반환:** 값이 아니라 텐서 원소의 저장 형식으로 사용됩니다.
- **이 셀에서 쓰는 이유:** 일반적인 학습 실수 dtype입니다.
- **주의점:** 연산마다 허용 dtype이 다르고 변환 시 정밀도나 메모리가 달라집니다.
- **공식 문서:** [torch.float32](https://docs.pytorch.org/docs/stable/tensor_attributes.html#torch.dtype)

</details>

<details>
<summary><code>tensor.sum(dim=None, keepdim=False, dtype=None)</code></summary>

#### `tensor.sum(dim=None, keepdim=False, dtype=None)`

- **역할:** 지정 축의 원소 합을 계산합니다.
- **입력·반환:** 텐서와 축 설정을 받고 축약된 텐서를 반환합니다.
- **이 셀에서 쓰는 이유:** 확률 질량, mask 개수, message aggregation을 계산합니다.
- **주의점:** 정수 overflow와 축약 뒤 shape 변화를 확인해야 합니다.

##### 관련 수식과 코드 연결

$$
S=\sum_{i=1}^{N}x_i
$$

- **기호:** $N$은 선택 축의 원소 수입니다.
- **수식 → 코드:** `dim` 또는 `axis`의 모든 원소를 더해 축약합니다.
- **직관:** count, 확률 질량, 이웃 message처럼 누적량을 계산합니다.
- **shape 확인:** 선택한 축은 제거되거나 `keepdim=True`이면 크기 1로 남습니다.
- **공식 문서:** [Tensor.sum](https://docs.pytorch.org/docs/stable/generated/torch.sum.html)

</details>

</details>
<!-- api-explanations:end -->

In [ ]:
def build_variable_sequences(frame, means, stds, max_history=12, stride=3):
    sequences, labels = [], []
    for _, group in frame.sort_values("timestamp").groupby("device_id", sort=False):
        values = ((group[FEATURES] - means) / stds).to_numpy(np.float32)
        anomaly = group["anomaly"].to_numpy(np.float32)
        for end in range(max_history, len(group) + 1, stride):
            length = 6 + ((end // stride) % 7)  # 6..12의 가변 길이
            sequences.append(torch.tensor(values[end - length : end]))
            labels.append(float(anomaly[end - length : end].max()))
    return sequences, torch.tensor(labels, dtype=torch.float32)


train_sequences, train_labels = build_variable_sequences(train_frame, means, stds)
test_sequences, test_labels = build_variable_sequences(test_frame, means, stds)
print(
    len(train_sequences),
    len(test_sequences),
    "positive train windows=",
    int(train_labels.sum()),
)

## 2) padding과 원래 길이

padding은 배치 텐서를 직사각형으로 만들 뿐 실제 관측값이 아닙니다. 원래 길이를
보존한 뒤 packing하거나 mask해야 모델이 padding을 신호로 학습하지 않습니다.

<!-- api-explanations:start -->
<!-- api-explanations:keys=torch.stack,DataLoader -->
<details>
<summary><strong>이 셀에서 처음 만나는 함수·API·수식 해설 (API 2개 · 수식 1개)</strong></summary>

> 아래 내용은 라이브러리 API의 의미를 설명합니다. 실습 정답 로직은 포함하지 않습니다.
> 수식은 별도 표기가 없으면 대표적인 기본 설정을 나타냅니다. `dim`, `reduction`, 가중치 같은 실제 인자에 따라 세부 형태가 달라질 수 있습니다.

<details>
<summary><code>torch.stack(tensors, dim=0)</code></summary>

#### `torch.stack(tensors, dim=0)`

- **역할:** 새 축을 만들어 텐서를 쌓습니다.
- **입력·반환:** 텐서와 축·연산 설정을 받고 변환된 텐서를 반환합니다.
- **이 셀에서 쓰는 이유:** 수식의 축 관계와 batch 연산을 코드로 직접 표현합니다.
- **주의점:** 입력 shape과 broadcasting 규칙을 셀의 shape assertion으로 확인해야 합니다.

##### 관련 수식과 코드 연결

$$
\operatorname{shape}(Y)=(d_1,\ldots,d_{a-1},R,d_a,\ldots,d_K)
$$

- **기호:** $R$은 쌓는 tensor 개수이고 $a$는 새 축이 들어갈 위치입니다.
- **수식 → 코드:** `stack`은 각 입력을 새 index 하나로 배치해 새로운 축을 만듭니다.
- **직관:** 연결할 기존 축을 고르는 `cat`과 달리 축의 개수가 하나 늘어납니다.
- **shape 확인:** 모든 입력 shape이 완전히 같아야 합니다.
- **공식 문서:** [torch.stack](https://docs.pytorch.org/docs/stable/generated/torch.stack.html)

</details>

<details>
<summary><code>DataLoader(dataset, batch_size=1, shuffle=False, ...)</code></summary>

#### `DataLoader(dataset, batch_size=1, shuffle=False, ...)`

- **역할:** dataset에서 mini-batch를 반복 생성합니다.
- **입력·반환:** dataset과 batching 설정을 받고 iterable loader를 반환합니다.
- **이 셀에서 쓰는 이유:** 학습 데이터를 batch 단위로 섞고 모델에 공급합니다.
- **주의점:** Windows 다중 worker는 진입점 보호가 필요하며 마지막 batch 크기도 확인합니다.
- **공식 문서:** [DataLoader](https://docs.pytorch.org/docs/stable/data.html#torch.utils.data.DataLoader)

</details>

</details>
<!-- api-explanations:end -->

In [ ]:
class SequenceDataset(Dataset):
    def __init__(self, sequences, labels):
        self.sequences, self.labels = sequences, labels

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, index):
        return self.sequences[index], self.labels[index]


def sequence_collate(batch):
    sequences, labels = zip(*batch)
    lengths = torch.tensor([len(sequence) for sequence in sequences], dtype=torch.long)
    padded = pad_sequence(sequences, batch_first=True, padding_value=0.0)
    return padded, lengths, torch.stack(labels)


train_loader = DataLoader(
    SequenceDataset(train_sequences, train_labels),
    batch_size=64,
    shuffle=True,
    collate_fn=sequence_collate,
)
test_loader = DataLoader(
    SequenceDataset(test_sequences, test_labels),
    batch_size=128,
    shuffle=False,
    collate_fn=sequence_collate,
)
batch_x, batch_lengths, batch_y = next(iter(train_loader))
print(
    batch_x.shape,
    batch_lengths.shape,
    batch_y.shape,
    batch_lengths.min().item(),
    batch_lengths.max().item(),
)

In [ ]:
assert batch_x.ndim == 3 and batch_x.shape[2] == len(FEATURES)
assert int(batch_lengths.max()) == batch_x.shape[1]
assert torch.all((batch_x.abs().sum(dim=2) != 0).sum(dim=1) <= batch_lengths)
print("padded batch contract: OK")

## 3) 하나의 wrapper로 RNN/GRU/LSTM 비교

세 모듈은 공통적으로 `(output, hidden)`을 반환합니다. LSTM만 hidden이 `(h, c)`
튜플이며, 다층/양방향 모델의 `h` shape는 `(layers × directions, batch, hidden)`입니다.

<!-- api-explanations:start -->
<!-- api-explanations:keys=nn.RNN,nn.GRU,nn.LSTM,nn.Linear,Tensor.cpu -->
<details>
<summary><strong>이 셀에서 처음 만나는 함수·API·수식 해설 (API 5개 · 수식 4개)</strong></summary>

> 아래 내용은 라이브러리 API의 의미를 설명합니다. 실습 정답 로직은 포함하지 않습니다.
> 수식은 별도 표기가 없으면 대표적인 기본 설정을 나타냅니다. `dim`, `reduction`, 가중치 같은 실제 인자에 따라 세부 형태가 달라질 수 있습니다.

<details>
<summary><code>nn.RNN(input_size, hidden_size, nonlinearity=&#x27;tanh&#x27;, ...)</code></summary>

#### `nn.RNN(input_size, hidden_size, nonlinearity='tanh', ...)`

- **역할:** 이전 hidden state와 현재 입력을 결합하는 기본 recurrent 층을 만듭니다.
- **입력·반환:** 입력·hidden 크기와 설정을 받고 sequence module을 반환합니다.
- **이 셀에서 쓰는 이유:** 단순 RNN의 시간축 상태 전달을 GRU·LSTM과 비교합니다.
- **주의점:** 긴 수열에서는 gradient 소실·폭주가 쉬우며 hidden 축 계약을 확인해야 합니다.

##### 관련 수식과 코드 연결

$$
h_t=\tanh(W_{ih}x_t+b_{ih}+W_{hh}h_{t-1}+b_{hh})
$$

- **기호:** $x_t$는 현재 입력, $h_{t-1},h_t$는 이전·현재 hidden state입니다.
- **수식 → 코드:** RNN module이 sequence 순서대로 같은 weight를 반복 적용합니다.
- **직관:** 현재 상태가 이전 상태와 새 입력을 압축해 다음 시간으로 전달됩니다.
- **shape 확인:** `batch_first=True`이면 입력 `[B,T,D]`, 출력 `[B,T,H]`, hidden `[L,B,H]`입니다.
- **공식 문서:** [nn.RNN](https://docs.pytorch.org/docs/stable/generated/torch.nn.RNN.html)

</details>

<details>
<summary><code>nn.GRU(input_size, hidden_size, ...)</code></summary>

#### `nn.GRU(input_size, hidden_size, ...)`

- **역할:** gated recurrent unit 층을 만듭니다.
- **입력·반환:** 구조 hyperparameter를 받고 호출 가능한 `nn.Module` 또는 파라미터를 반환합니다.
- **이 셀에서 쓰는 이유:** 논문의 학습 가능한 변환을 명시적인 구성 요소로 조립합니다.
- **주의점:** 입·출력 shape, bias 여부, 등록 상태가 논문 수식과 같은지 확인해야 합니다.

##### 관련 수식과 코드 연결

$$
\begin{aligned}r_t&=\sigma(W_{ir}x_t+b_{ir}+W_{hr}h_{t-1}+b_{hr}),\\z_t&=\sigma(W_{iz}x_t+b_{iz}+W_{hz}h_{t-1}+b_{hz}),\\n_t&=\tanh(W_{in}x_t+b_{in}+r_t\odot(W_{hn}h_{t-1}+b_{hn})),\\h_t&=(1-z_t)\odot n_t+z_t\odot h_{t-1}\end{aligned}
$$

- **기호:** $r_t,z_t$는 reset·update gate, $n_t$는 후보, $h_t$는 hidden state입니다.
- **수식 → 코드:** `nn.GRU`가 time step마다 이 네 식을 계산하고 hidden state를 다음 step으로 넘깁니다.
- **직관:** gate가 과거 정보를 얼마나 지우고 유지할지 학습합니다.
- **shape 확인:** `batch_first=True`이면 입력 `[B,T,D]`, 출력 `[B,T,H]`, 마지막 hidden `[L,B,H]`입니다.
- **공식 문서:** [nn.GRU](https://docs.pytorch.org/docs/stable/generated/torch.nn.GRU.html)

</details>

<details>
<summary><code>nn.LSTM(input_size, hidden_size, ...)</code></summary>

#### `nn.LSTM(input_size, hidden_size, ...)`

- **역할:** input·forget·output gate와 cell state를 가진 recurrent 층을 만듭니다.
- **입력·반환:** 입력·hidden 크기와 설정을 받고 sequence module을 반환합니다.
- **이 셀에서 쓰는 이유:** 장기 정보를 cell state로 보존하는 sequence encoder·decoder를 구현합니다.
- **주의점:** hidden state와 cell state 두 개를 전달하며 `batch_first`가 둘의 shape은 바꾸지 않습니다.

##### 관련 수식과 코드 연결

$$
\begin{aligned}i_t&=\sigma(W_{ii}x_t+b_{ii}+W_{hi}h_{t-1}+b_{hi}),\\f_t&=\sigma(W_{if}x_t+b_{if}+W_{hf}h_{t-1}+b_{hf}),\\g_t&=\tanh(W_{ig}x_t+b_{ig}+W_{hg}h_{t-1}+b_{hg}),\\o_t&=\sigma(W_{io}x_t+b_{io}+W_{ho}h_{t-1}+b_{ho}),\\c_t&=f_t\odot c_{t-1}+i_t\odot g_t,\qquad h_t=o_t\odot\tanh(c_t)\end{aligned}
$$

- **기호:** $i,f,o$는 input·forget·output gate, $c_t$는 cell state입니다.
- **수식 → 코드:** LSTM module이 각 time step에서 네 gate와 cell·hidden update를 계산합니다.
- **직관:** cell state가 정보를 더하기 경로로 전달해 기본 RNN보다 긴 의존성을 보존합니다.
- **shape 확인:** 입력·출력은 RNN과 같고 마지막 상태는 `(h_n, c_n)` 두 `[L,B,H]` 텐서입니다.
- **공식 문서:** [nn.LSTM](https://docs.pytorch.org/docs/stable/generated/torch.nn.LSTM.html)

</details>

<details>
<summary><code>nn.Linear(in_features, out_features, bias=True)</code></summary>

#### `nn.Linear(in_features, out_features, bias=True)`

- **역할:** 완전연결 affine 변환을 만듭니다.
- **입력·반환:** 구조 hyperparameter를 받고 호출 가능한 `nn.Module` 또는 파라미터를 반환합니다.
- **이 셀에서 쓰는 이유:** 논문의 학습 가능한 변환을 명시적인 구성 요소로 조립합니다.
- **주의점:** 입·출력 shape, bias 여부, 등록 상태가 논문 수식과 같은지 확인해야 합니다.

##### 관련 수식과 코드 연결

$$
y=xW^{\mathsf T}+b
$$

- **기호:** $x$는 입력, $W$는 weight, $b$는 bias, $y$는 출력입니다.
- **수식 → 코드:** `in_features`와 `out_features`가 각각 $W$의 두 축을 결정합니다.
- **직관:** 각 출력 feature는 모든 입력 feature의 학습 가능한 가중합입니다.
- **shape 확인:** `[..., D_in] -> [..., D_out]`, weight는 `[D_out, D_in]`입니다.
- **공식 문서:** [nn.Linear](https://docs.pytorch.org/docs/stable/generated/torch.nn.Linear.html)

</details>

<details>
<summary><code>tensor.cpu()</code></summary>

#### `tensor.cpu()`

- **역할:** 텐서를 CPU 메모리로 옮깁니다.
- **입력·반환:** 텐서를 받아 CPU 텐서를 반환합니다.
- **이 셀에서 쓰는 이유:** NumPy 변환이나 CPU 기반 평가 도구에 값을 전달합니다.
- **주의점:** GPU 텐서 이동은 동기화·복사 비용을 만들 수 있습니다.
- **공식 문서:** [Tensor.cpu](https://docs.pytorch.org/docs/stable/generated/torch.Tensor.cpu.html)

</details>

</details>
<!-- api-explanations:end -->

In [ ]:
class SequenceClassifier(nn.Module):
    def __init__(
        self, cell: str, input_dim: int, hidden_dim: int = 24, num_layers: int = 1
    ):
        super().__init__()
        cells = {"RNN": nn.RNN, "GRU": nn.GRU, "LSTM": nn.LSTM}
        if cell not in cells:
            raise ValueError(f"unknown cell: {cell}")
        self.cell_name = cell
        self.recurrent = cells[cell](
            input_dim, hidden_dim, num_layers=num_layers, batch_first=True
        )
        self.head = nn.Linear(hidden_dim, 1)

    def forward(self, x, lengths, return_hidden=False):
        packed = pack_padded_sequence(
            x, lengths.cpu(), batch_first=True, enforce_sorted=False
        )
        _, hidden = self.recurrent(packed)
        h = hidden[0] if isinstance(hidden, tuple) else hidden
        logits = self.head(h[-1]).squeeze(1)
        return (logits, hidden) if return_hidden else logits

In [ ]:
for cell in ("RNN", "GRU", "LSTM"):
    probe = SequenceClassifier(cell, len(FEATURES), hidden_dim=16)
    logits, hidden = probe(batch_x[:5], batch_lengths[:5], return_hidden=True)
    if isinstance(hidden, tuple):
        hidden_shapes = [tuple(item.shape) for item in hidden]
    else:
        hidden_shapes = tuple(hidden.shape)
    print(cell, "logits", tuple(logits.shape), "hidden", hidden_shapes)
    assert logits.shape == (5,)

## 4) gradient clipping을 포함한 학습 루프

긴 시퀀스의 반복 곱은 gradient exploding을 일으킬 수 있습니다. 역전파 후
optimizer step 전에 `clip_grad_norm_`을 호출하고, 반환되는 clipping 전 norm을 기록합니다.

<!-- api-explanations:start -->
<!-- api-explanations:keys=dataclass,Module.train,Tensor.to,Optimizer.zero_grad,Tensor.backward,clip_grad_norm_,Module.parameters,Optimizer.step,Tensor.item,torch.inference_mode,Module.eval,torch.cat -->
<details>
<summary><strong>이 셀에서 처음 만나는 함수·API·수식 해설 (API 12개 · 수식 4개)</strong></summary>

> 아래 내용은 라이브러리 API의 의미를 설명합니다. 실습 정답 로직은 포함하지 않습니다.
> 수식은 별도 표기가 없으면 대표적인 기본 설정을 나타냅니다. `dim`, `reduction`, 가중치 같은 실제 인자에 따라 세부 형태가 달라질 수 있습니다.

<details>
<summary><code>@dataclass(frozen=False, slots=False, ...)</code></summary>

#### `@dataclass(frozen=False, slots=False, ...)`

- **역할:** type annotation을 바탕으로 초기화·표현 메서드 등을 생성합니다.
- **입력·반환:** class decorator 설정을 받고 변환된 class를 반환합니다.
- **이 셀에서 쓰는 이유:** 실험 설정과 결과 record의 필드를 명시적으로 정의합니다.
- **주의점:** mutable 기본값은 `default_factory`를 쓰고 equality·frozen 의미를 확인합니다.
- **공식 문서:** [dataclass](https://docs.python.org/3/library/dataclasses.html#dataclasses.dataclass)

</details>

<details>
<summary><code>model.train(mode=True)</code></summary>

#### `model.train(mode=True)`

- **역할:** 모듈과 하위 모듈을 학습 모드로 전환합니다.
- **입력·반환:** 불리언 모드를 받고 모듈 자신을 반환합니다.
- **이 셀에서 쓰는 이유:** Dropout과 BatchNorm이 학습 시 동작을 사용하도록 합니다.
- **주의점:** gradient를 켜는 함수가 아니므로 `no_grad`와 독립적입니다.
- **공식 문서:** [Module.train](https://docs.pytorch.org/docs/stable/generated/torch.nn.Module.html)

</details>

<details>
<summary><code>tensor.to(device=None, dtype=None)</code></summary>

#### `tensor.to(device=None, dtype=None)`

- **역할:** 텐서를 지정 장치나 dtype으로 변환합니다.
- **입력·반환:** 장치·dtype을 받고 변환된 텐서를 반환합니다.
- **이 셀에서 쓰는 이유:** 모델과 입력의 장치·정밀도를 일치시킵니다.
- **주의점:** 원본과 조건이 같으면 복사 없이 같은 텐서를 돌려줄 수 있습니다.
- **공식 문서:** [Tensor.to](https://docs.pytorch.org/docs/stable/generated/torch.Tensor.to.html)

</details>

<details>
<summary><code>optimizer.zero_grad(set_to_none=True)</code></summary>

#### `optimizer.zero_grad(set_to_none=True)`

- **역할:** 파라미터에 누적된 gradient를 비웁니다.
- **입력·반환:** 초기화 방식을 받고 반환값은 없습니다.
- **이 셀에서 쓰는 이유:** 현재 mini-batch의 gradient만으로 update하도록 학습 단계를 시작합니다.
- **주의점:** 호출하지 않으면 PyTorch gradient가 기본적으로 누적됩니다.
- **공식 문서:** [Optimizer.zero_grad](https://docs.pytorch.org/docs/stable/generated/torch.optim.Optimizer.zero_grad.html)

</details>

<details>
<summary><code>loss.backward()</code></summary>

#### `loss.backward()`

- **역할:** 스칼라 loss에서 계산 그래프를 거슬러 gradient를 계산합니다.
- **입력·반환:** 스칼라 텐서에서는 보통 인자 없이 호출하며 반환값은 없습니다.
- **이 셀에서 쓰는 이유:** optimizer update에 필요한 각 파라미터의 `.grad`를 채웁니다.
- **주의점:** 같은 그래프를 다시 역전파하려면 별도 보존 설정이 필요합니다.

##### 관련 수식과 코드 연결

$$
\frac{\partial L}{\partial x}=\frac{\partial L}{\partial y}\frac{\partial y}{\partial x},\qquad x.\mathrm{grad}\leftarrow x.\mathrm{grad}+\nabla_xL
$$

- **기호:** $L$은 scalar loss이고 $x$는 계산 그래프의 leaf parameter입니다.
- **수식 → 코드:** `backward()`는 연쇄법칙으로 gradient를 계산해 각 parameter의 `.grad`에 누적합니다.
- **직관:** 파라미터를 직접 바꾸는 호출이 아니라 update에 필요한 미분값을 준비하는 호출입니다.
- **shape 확인:** 각 `.grad`는 대응 parameter와 같은 shape이며 scalar가 아닌 출력에는 seed gradient가 필요합니다.
- **공식 문서:** [Tensor.backward](https://docs.pytorch.org/docs/stable/generated/torch.Tensor.backward.html)

</details>

<details>
<summary><code>clip_grad_norm_(parameters, max_norm, norm_type=2.0, ...)</code></summary>

#### `clip_grad_norm_(parameters, max_norm, norm_type=2.0, ...)`

- **역할:** 전체 gradient norm이 한계를 넘으면 같은 비율로 축소합니다.
- **입력·반환:** 파라미터와 최대 norm을 받고 clipping 전 전체 norm을 반환합니다.
- **이 셀에서 쓰는 이유:** RNN·Transformer·RL 학습의 폭주하는 update를 제한합니다.
- **주의점:** `backward()` 뒤 `step()` 전에 호출해야 하며 근본 원인을 없애지는 않습니다.

##### 관련 수식과 코드 연결

$$
g' = g\cdot\min\!\left(1,\frac{c}{\lVert g\rVert_p+\epsilon}\right)
$$

- **기호:** $g$는 모든 parameter gradient를 합친 벡터, $c$는 `max_norm`입니다.
- **수식 → 코드:** 전체 norm이 한계를 넘을 때만 모든 gradient를 같은 비율로 줄입니다.
- **직관:** 방향은 유지하면서 update 크기만 제한합니다.
- **shape 확인:** 각 parameter의 gradient shape은 유지됩니다.
- **공식 문서:** [clip_grad_norm_](https://docs.pytorch.org/docs/stable/generated/torch.nn.utils.clip_grad_norm_.html)

</details>

<details>
<summary><code>model.parameters(recurse=True)</code></summary>

#### `model.parameters(recurse=True)`

- **역할:** 모듈에 등록된 학습 파라미터 iterator를 반환합니다.
- **입력·반환:** 하위 모듈 포함 여부를 받고 `Parameter` iterator를 반환합니다.
- **이 셀에서 쓰는 이유:** optimizer가 갱신할 weight와 bias를 전달합니다.
- **주의점:** iterator는 한 번 소비되며 buffer와 일반 tensor attribute는 포함하지 않습니다.
- **공식 문서:** [Module.parameters](https://docs.pytorch.org/docs/stable/generated/torch.nn.Module.html)

</details>

<details>
<summary><code>optimizer.step(closure=None)</code></summary>

#### `optimizer.step(closure=None)`

- **역할:** 현재 gradient와 optimizer 규칙으로 파라미터를 갱신합니다.
- **입력·반환:** 선택적 closure를 받고 optimizer에 따라 loss 또는 `None`을 반환합니다.
- **이 셀에서 쓰는 이유:** 역전파로 계산한 gradient를 실제 학습 update에 반영합니다.
- **주의점:** 일반적으로 `zero_grad`, forward, `backward` 다음에 호출합니다.

##### 관련 수식과 코드 연결

$$
\theta_{t+1}=\mathcal{U}(\theta_t,\nabla_\theta L_t,s_t;\eta)
$$

- **기호:** $\mathcal U$는 optimizer update, $s_t$는 momentum·moment 상태입니다.
- **수식 → 코드:** `step()`은 이미 `.grad`에 저장된 값을 읽어 선택한 optimizer 규칙을 적용합니다.
- **직관:** gradient 계산과 parameter 갱신을 분리하면 학습 lifecycle을 정확히 추적할 수 있습니다.
- **shape 확인:** parameter shape은 유지되고 값과 optimizer state만 바뀝니다.
- **공식 문서:** [Optimizer.step](https://docs.pytorch.org/docs/stable/generated/torch.optim.Optimizer.step.html)

</details>

<details>
<summary><code>tensor.item()</code></summary>

#### `tensor.item()`

- **역할:** 원소 하나짜리 텐서에서 Python 숫자를 꺼냅니다.
- **입력·반환:** 단일 원소 텐서를 받고 Python 숫자를 반환합니다.
- **이 셀에서 쓰는 이유:** loss나 metric을 출력·기록 가능한 값으로 바꿉니다.
- **주의점:** 여러 원소 텐서에는 쓸 수 없고 GPU에서는 동기화를 일으킬 수 있습니다.
- **공식 문서:** [Tensor.item](https://docs.pytorch.org/docs/stable/generated/torch.Tensor.item.html)

</details>

<details>
<summary><code>torch.inference_mode(mode=True)</code></summary>

#### `torch.inference_mode(mode=True)`

- **역할:** 추론 전용으로 autograd 관련 추적을 더 강하게 비활성화합니다.
- **입력·반환:** 불리언 모드를 받고 context manager 또는 decorator로 동작합니다.
- **이 셀에서 쓰는 이유:** 순수 평가 구간의 오버헤드를 줄입니다.
- **주의점:** 이 모드에서 만든 텐서는 이후 gradient 계산에 쓰기 어려울 수 있습니다.
- **공식 문서:** [torch.inference_mode](https://docs.pytorch.org/docs/stable/generated/torch.inference_mode.html)

</details>

<details>
<summary><code>model.eval()</code></summary>

#### `model.eval()`

- **역할:** 모듈과 하위 모듈을 평가 모드로 전환합니다.
- **입력·반환:** 인자 없이 호출하고 모듈 자신을 반환합니다.
- **이 셀에서 쓰는 이유:** Dropout을 끄고 BatchNorm의 저장 통계를 사용하게 합니다.
- **주의점:** gradient 기록은 끄지 않으므로 필요하면 `no_grad`도 사용합니다.
- **공식 문서:** [Module.eval](https://docs.pytorch.org/docs/stable/generated/torch.nn.Module.html)

</details>

<details>
<summary><code>torch.cat(tensors, dim=0)</code></summary>

#### `torch.cat(tensors, dim=0)`

- **역할:** 기존 축을 따라 텐서를 연결합니다.
- **입력·반환:** 텐서와 축·연산 설정을 받고 변환된 텐서를 반환합니다.
- **이 셀에서 쓰는 이유:** 수식의 축 관계와 batch 연산을 코드로 직접 표현합니다.
- **주의점:** 입력 shape과 broadcasting 규칙을 셀의 shape assertion으로 확인해야 합니다.

##### 관련 수식과 코드 연결

$$
d_{\mathrm{out}}^{(a)}=\sum_{r=1}^{R}d_r^{(a)}
$$

- **기호:** $a$는 연결할 축이고 $R$은 입력 tensor 개수입니다.
- **수식 → 코드:** `dim=a` 축의 길이만 더하고 나머지 축은 그대로 유지합니다.
- **직관:** 이미 존재하는 축을 길게 이어 붙이는 연산입니다.
- **shape 확인:** 연결 축을 제외한 모든 shape과 dtype·device가 같아야 합니다.
- **공식 문서:** [torch.cat](https://docs.pytorch.org/docs/stable/generated/torch.cat.html)

</details>

</details>
<!-- api-explanations:end -->

In [ ]:
@dataclass
class EpochStats:
    loss: float
    grad_norm: float


def train_epoch(model, loader, optimizer, loss_fn, max_norm=1.0):
    model.train()
    total_loss, max_grad = 0.0, 0.0
    for x, lengths, y in loader:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad(set_to_none=True)
        loss = loss_fn(model(x, lengths), y)
        loss.backward()
        grad_norm = clip_grad_norm_(model.parameters(), max_norm=max_norm)
        optimizer.step()
        total_loss += loss.item() * len(y)
        max_grad = max(max_grad, float(grad_norm))
    return EpochStats(total_loss / len(loader.dataset), max_grad)


@torch.inference_mode()
def evaluate_classifier(model, loader):
    model.eval()
    probabilities, labels = [], []
    for x, lengths, y in loader:
        probabilities.append(model(x.to(device), lengths).sigmoid().cpu())
        labels.append(y)
    probabilities = torch.cat(probabilities).numpy()
    labels = torch.cat(labels).numpy().astype(int)
    return (
        f1_score(labels, probabilities >= 0.5, zero_division=0),
        probabilities,
        labels,
    )

In [ ]:
positive = float(train_labels.sum())
negative = float(len(train_labels) - positive)
pos_weight = torch.tensor([negative / max(positive, 1.0)], device=device)
classification_loss = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
print("pos_weight:", round(pos_weight.item(), 2))

<!-- api-explanations:start -->
<!-- api-explanations:keys=torch.optim.Adam -->
<details>
<summary><strong>이 셀에서 처음 만나는 함수·API·수식 해설 (API 1개 · 수식 1개)</strong></summary>

> 아래 내용은 라이브러리 API의 의미를 설명합니다. 실습 정답 로직은 포함하지 않습니다.
> 수식은 별도 표기가 없으면 대표적인 기본 설정을 나타냅니다. `dim`, `reduction`, 가중치 같은 실제 인자에 따라 세부 형태가 달라질 수 있습니다.

<details>
<summary><code>torch.optim.Adam(params, lr=1e-3, ...)</code></summary>

#### `torch.optim.Adam(params, lr=1e-3, ...)`

- **역할:** 1·2차 gradient 모멘트를 추적하는 Adam optimizer를 만듭니다.
- **입력·반환:** 파라미터와 학습률 등을 받고 optimizer 객체를 반환합니다.
- **이 셀에서 쓰는 이유:** 노트북의 trainable parameter를 반복 update합니다.
- **주의점:** weight decay의 해석은 AdamW와 다르며 학습률이 결과에 크게 작용합니다.

##### 관련 수식과 코드 연결

$$
\begin{aligned}m_t&=\beta_1m_{t-1}+(1-\beta_1)g_t,\\v_t&=\beta_2v_{t-1}+(1-\beta_2)g_t^2,\\\theta_t&=\theta_{t-1}-\eta\frac{\hat m_t}{\sqrt{\hat v_t}+\epsilon}\end{aligned}
$$

- **기호:** $g_t$는 gradient, $m_t,v_t$는 1·2차 모멘트, $\eta$는 학습률입니다.
- **수식 → 코드:** `backward()`가 만든 gradient를 읽어 모멘트를 갱신한 뒤 `step()`에서 파라미터에 적용합니다.
- **직관:** 최근 gradient 방향과 크기를 함께 누적해 파라미터별 적응형 보폭을 만듭니다.
- **shape 확인:** 각 trainable parameter와 같은 shape의 모멘트 상태가 optimizer에 저장됩니다.
- **공식 문서:** [torch.optim.Adam](https://docs.pytorch.org/docs/stable/generated/torch.optim.Adam.html)

</details>

</details>
<!-- api-explanations:end -->

In [ ]:
comparison = {}
trained_models = {}
for cell in ("RNN", "GRU", "LSTM"):
    torch.manual_seed(SEED)
    model = SequenceClassifier(cell, len(FEATURES)).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=3e-3)
    history = []
    for epoch in range(3):
        stats = train_epoch(model, train_loader, optimizer, classification_loss)
        history.append(stats.loss)
    f1, probabilities, labels = evaluate_classifier(model, test_loader)
    comparison[cell] = {"final_loss": history[-1], "test_f1": f1}
    trained_models[cell] = model
    print(cell, comparison[cell])

<!-- api-explanations:start -->
<!-- api-explanations:keys=pd.DataFrame,np.isfinite -->
<details>
<summary><strong>이 셀에서 처음 만나는 함수·API·수식 해설 (API 2개 · 수식 0개)</strong></summary>

> 아래 내용은 라이브러리 API의 의미를 설명합니다. 실습 정답 로직은 포함하지 않습니다.
> 수식은 별도 표기가 없으면 대표적인 기본 설정을 나타냅니다. `dim`, `reduction`, 가중치 같은 실제 인자에 따라 세부 형태가 달라질 수 있습니다.

<details>
<summary><code>pd.DataFrame(data=None, index=None, columns=None, ...)</code></summary>

#### `pd.DataFrame(data=None, index=None, columns=None, ...)`

- **역할:** 행·열 구조의 DataFrame을 만듭니다.
- **입력·반환:** 표 데이터와 열·그룹 설정을 받고 DataFrame, Series 또는 집계 객체를 반환합니다.
- **이 셀에서 쓰는 이유:** 학습 전 데이터 분포·결측·중복과 분할 기준을 점검합니다.
- **주의점:** dtype 추론, 결측값, index 보존과 grouping 후 shape 변화를 확인해야 합니다.
- **공식 문서:** [pd.DataFrame](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.html)

</details>

<details>
<summary><code>np.isfinite(x)</code></summary>

#### `np.isfinite(x)`

- **역할:** 각 값이 NaN이나 무한대가 아닌 유한수인지 검사합니다.
- **입력·반환:** 배열형 입력을 받고 같은 shape의 불리언 배열을 반환합니다.
- **이 셀에서 쓰는 이유:** 학습 값이나 전처리 결과가 수치적으로 정상인지 검증합니다.
- **주의점:** 유한하다는 사실만 확인하며 값의 크기가 적절하다는 뜻은 아닙니다.
- **공식 문서:** [np.isfinite](https://numpy.org/doc/stable/reference/generated/numpy.isfinite.html)

</details>

</details>
<!-- api-explanations:end -->

In [ ]:
result_table = pd.DataFrame(comparison).T.sort_values("test_f1", ascending=False)
print(result_table)
assert set(result_table.index) == {"RNN", "GRU", "LSTM"}
assert np.isfinite(result_table.to_numpy()).all()

### 비교를 해석할 때 주의할 점

작은 데이터에서 한 번의 seed로 얻은 순위는 모델의 보편적 우열이 아닙니다.
RNN은 단순하지만 장기 의존성에 취약하고, GRU는 gate가 적어 가벼우며, LSTM은
cell state를 별도로 유지합니다. 반복 실험의 평균·편차와 latency를 함께 비교하세요.

## 5) padding 값 불변성 확인

packing이 올바르면 실제 길이 뒤의 padding 값을 크게 바꿔도 logits는 변하지 않습니다.
이 테스트는 mask/length 누락 버그를 매우 잘 찾아냅니다.

In [ ]:
best_name = result_table.index[0]
best_model = trained_models[best_name].eval()
x = batch_x[:8].to(device)
lengths = batch_lengths[:8]
changed_padding = x.clone()
for row, length in enumerate(lengths.tolist()):
    changed_padding[row, length:] = 999.0
with torch.inference_mode():
    original_logits = best_model(x, lengths)
    changed_logits = best_model(changed_padding, lengths)
max_difference = (original_logits - changed_logits).abs().max().item()
print("max padding-induced difference:", max_difference)
assert max_difference < 1e-5

## 6) checkpoint 저장과 복원

최소 checkpoint에는 모델 종류/차원, `state_dict`, 전처리 통계, feature 순서가 있어야
합니다. optimizer 상태와 epoch까지 저장하면 중단된 학습도 이어갈 수 있습니다.

<!-- api-explanations:start -->
<!-- api-explanations:keys=torch.allclose -->
<details>
<summary><strong>이 셀에서 처음 만나는 함수·API·수식 해설 (API 1개 · 수식 1개)</strong></summary>

> 아래 내용은 라이브러리 API의 의미를 설명합니다. 실습 정답 로직은 포함하지 않습니다.
> 수식은 별도 표기가 없으면 대표적인 기본 설정을 나타냅니다. `dim`, `reduction`, 가중치 같은 실제 인자에 따라 세부 형태가 달라질 수 있습니다.

<details>
<summary><code>torch.allclose(input, other, rtol=1e-5, atol=1e-8, equal_nan=False)</code></summary>

#### `torch.allclose(input, other, rtol=1e-5, atol=1e-8, equal_nan=False)`

- **역할:** 두 텐서의 모든 원소가 허용 오차 안에서 가까운지 검사합니다.
- **입력·반환:** 두 텐서와 오차 설정을 받고 Python `bool` 하나를 반환합니다.
- **이 셀에서 쓰는 이유:** 장치나 구현을 바꾼 뒤 수치 결과가 사실상 같은지 확인합니다.
- **주의점:** 정확한 동일성 검사가 아니며 작은 값에서는 `atol`을 직접 정해야 합니다.

##### 관련 수식과 코드 연결

$$
\lvert a_i-b_i\rvert \le \mathrm{atol}+\mathrm{rtol}\,\lvert b_i\rvert
$$

- **기호:** $a_i$는 계산값, $b_i$는 기준값, `atol`은 절대 오차, `rtol`은 상대 오차 허용치입니다.
- **수식 → 코드:** `isclose`는 이 부등식을 원소마다 계산하고, `allclose`는 모든 결과에 논리 AND를 적용합니다.
- **직관:** 완전히 같은 비트인지가 아니라 실험 목적에 충분히 가까운지를 묻는 검사입니다.
- **shape 확인:** 두 입력은 broadcasting 가능해야 합니다. `isclose`는 broadcast shape, `allclose`는 scalar `bool`을 반환합니다.
- **공식 문서:** [torch.allclose](https://docs.pytorch.org/docs/stable/generated/torch.allclose.html)

</details>

</details>
<!-- api-explanations:end -->

In [ ]:
checkpoint_path = ARTIFACTS / "rnn_sequence_classifier_solution.pt"
checkpoint = {
    "cell": best_name,
    "input_dim": len(FEATURES),
    "hidden_dim": 24,
    "state_dict": best_model.state_dict(),
    "features": FEATURES,
    "means": means.to_dict(),
    "stds": stds.to_dict(),
}
torch.save(checkpoint, checkpoint_path)

loaded = torch.load(checkpoint_path, map_location=device, weights_only=True)
restored = SequenceClassifier(
    loaded["cell"], loaded["input_dim"], loaded["hidden_dim"]
).to(device)
restored.load_state_dict(loaded["state_dict"])
restored.eval()
with torch.inference_mode():
    restored_logits = restored(x, lengths)
assert torch.allclose(original_logits, restored_logits)
print("checkpoint restored:", checkpoint_path)

## 7) 다음 온도 forecasting

분류와 달리 연속값을 예측하므로 MSE로 학습하고 MAE로 평가합니다. 각 window 직후의
`next_temperature`를 target으로 하며, target 역시 train 평균/표준편차로 정규화합니다.

In [ ]:
def make_forecast_samples(frame, feature_means, feature_stds, history=12, stride=3):
    xs, ys = [], []
    target_mean = float(train_frame["next_temperature"].mean())
    target_std = float(train_frame["next_temperature"].std())
    for _, group in frame.sort_values("timestamp").groupby("device_id", sort=False):
        values = ((group[FEATURES] - feature_means) / feature_stds).to_numpy(np.float32)
        targets = group["next_temperature"].to_numpy(np.float32)
        for end in range(history, len(group), stride):
            xs.append(torch.tensor(values[end - history : end]))
            ys.append((targets[end - 1] - target_mean) / target_std)
    return (
        torch.stack(xs),
        torch.tensor(ys, dtype=torch.float32),
        target_mean,
        target_std,
    )


forecast_train_x, forecast_train_y, target_mean, target_std = make_forecast_samples(
    train_frame, means, stds
)
forecast_test_x, forecast_test_y, _, _ = make_forecast_samples(test_frame, means, stds)
print(forecast_train_x.shape, forecast_train_y.shape)

<!-- api-explanations:start -->
<!-- api-explanations:keys=F.mse_loss -->
<details>
<summary><strong>이 셀에서 처음 만나는 함수·API·수식 해설 (API 1개 · 수식 1개)</strong></summary>

> 아래 내용은 라이브러리 API의 의미를 설명합니다. 실습 정답 로직은 포함하지 않습니다.
> 수식은 별도 표기가 없으면 대표적인 기본 설정을 나타냅니다. `dim`, `reduction`, 가중치 같은 실제 인자에 따라 세부 형태가 달라질 수 있습니다.

<details>
<summary><code>F.mse_loss(input, target, reduction=&#x27;mean&#x27;)</code></summary>

#### `F.mse_loss(input, target, reduction='mean')`

- **역할:** 예측과 target의 제곱 오차를 계산합니다.
- **입력·반환:** 예측·target과 설정을 받고 축약된 loss 또는 같은 shape의 loss를 반환합니다.
- **이 셀에서 쓰는 이유:** 논문의 objective를 수치화해 역전파 시작점으로 사용합니다.
- **주의점:** 입력 형식, target dtype·shape, `reduction` 의미를 반드시 확인해야 합니다.

##### 관련 수식과 코드 연결

$$
\mathcal{L}_{MSE}=\frac{1}{N}\sum_{i=1}^{N}(\hat y_i-y_i)^2
$$

- **기호:** $\hat y_i$는 예측, $y_i$는 target입니다.
- **수식 → 코드:** `reduction='mean'`이 모든 원소의 제곱 오차를 평균냅니다.
- **직관:** 큰 오차를 제곱해 더 강하게 벌점하므로 회귀와 reconstruction에 자주 씁니다.
- **shape 확인:** 예측과 target은 같거나 broadcast 가능한 shape이어야 합니다.
- **공식 문서:** [F.mse_loss](https://docs.pytorch.org/docs/stable/generated/torch.nn.functional.mse_loss.html)

</details>

</details>
<!-- api-explanations:end -->

In [ ]:
class GRUForecaster(nn.Module):
    def __init__(self, input_dim, hidden_dim=24):
        super().__init__()
        self.gru = nn.GRU(input_dim, hidden_dim, batch_first=True)
        self.head = nn.Linear(hidden_dim, 1)

    def forward(self, x):
        _, hidden = self.gru(x)
        return self.head(hidden[-1]).squeeze(1)


forecast_model = GRUForecaster(len(FEATURES)).to(device)
forecast_optimizer = torch.optim.Adam(forecast_model.parameters(), lr=3e-3)
forecast_loader = DataLoader(
    torch.utils.data.TensorDataset(forecast_train_x, forecast_train_y),
    batch_size=64,
    shuffle=True,
)
for epoch in range(4):
    forecast_model.train()
    total = 0.0
    for bx, by in forecast_loader:
        bx, by = bx.to(device), by.to(device)
        forecast_optimizer.zero_grad(set_to_none=True)
        loss = nn.functional.mse_loss(forecast_model(bx), by)
        loss.backward()
        clip_grad_norm_(forecast_model.parameters(), 1.0)
        forecast_optimizer.step()
        total += loss.item() * len(by)
    print(
        f"forecast epoch {epoch + 1}: loss={total / len(forecast_loader.dataset):.4f}"
    )

In [ ]:
forecast_model.eval()
with torch.inference_mode():
    normalized_prediction = forecast_model(forecast_test_x.to(device)).cpu().numpy()
predicted_temperature = normalized_prediction * target_std + target_mean
actual_temperature = forecast_test_y.numpy() * target_std + target_mean
forecast_mae = mean_absolute_error(actual_temperature, predicted_temperature)
print("test MAE (°C):", round(forecast_mae, 3))
for actual, predicted in list(zip(actual_temperature, predicted_temperature))[:5]:
    print(f"actual={actual:6.2f}  predicted={predicted:6.2f}")
assert np.isfinite(forecast_mae)

## hidden state shape 기억법

- 입력(`batch_first=True`): `(batch, time, input_dim)`
- output: `(batch, time, directions × hidden_dim)`
- h: `(layers × directions, batch, hidden_dim)`
- LSTM: `(h, c)` 두 텐서를 반환
- packed 입력에서도 hidden은 각 샘플의 **실제 마지막 timestep**을 반영

In [ ]:
summary = {
    "classification_models": list(comparison),
    "best_cell": best_name,
    "padding_invariance": max_difference,
    "forecast_mae": float(forecast_mae),
    "checkpoint": str(checkpoint_path),
}
print(summary)
assert checkpoint_path.exists()
assert summary["padding_invariance"] < 1e-5

## 확장 과제

1. `num_layers=2`, dropout, bidirectional을 추가하고 hidden shape를 기록하세요.
2. 분류 threshold를 validation F1로 선택하고 test에는 한 번만 적용하세요.
3. forecasting을 한 시점이 아니라 다음 6개 시점 출력으로 바꾸세요.
4. gradient norm을 epoch별로 기록하여 clipping 전후 안정성을 비교하세요.

## 마무리

시퀀스 모델의 핵심은 셀 이름보다 **시간 분할, 길이 전달, hidden 선택, gradient 안정성,
재현 가능한 checkpoint**입니다. Transformer에서도 padding mask와 시간 누수 원칙은 그대로 이어집니다.